# CreditWise AI - Model Persistence and Inference Pipeline

## Objective

In the previous notebook, multiple machine learning models were trained and evaluated for credit default prediction. The final selected solution was an ensemble of LightGBM and XGBoost models, which achieved the best overall performance.

This notebook focuses on preparing the solution for deployment by:

- Retraining the final models on the complete training dataset
- Saving trained model artifacts
- Saving feature schema information
- Building a reusable prediction pipeline
- Generating business-friendly risk scores
- Defining risk categories
- Testing inference on sample applicants
- Preparing assets for API and dashboard integration

## Expected Outputs

By the end of this notebook, the following deployment-ready artifacts will be available:

- LightGBM Model
- XGBoost Model
- Ensemble Prediction Pipeline
- Feature Schema
- Risk Scoring Logic
- Deployment Assets

These artifacts will later be integrated into a FastAPI backend, PostgreSQL database, explainability module, and interactive dashboard.

In [ ]:
import pandas as pd
import numpy as np

import joblib

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

## Load Processed Dataset

The fully preprocessed dataset generated during feature engineering is loaded for final model training and deployment preparation.

In [ ]:
train_df = pd.read_csv(
    "../data/processed/train_processed.csv"
)

train_df.shape

## Remove Non-Predictive Identifier Columns

Previous experiments showed that customer identifier columns do not contribute meaningful predictive information and should not be used during deployment.

In [ ]:
train_df = train_df.drop(
    columns=["SK_ID_CURR"],
    errors="ignore"
)

train_df.shape

## Separate Features and Target

The dataset is separated into predictor variables and the target variable for model training.

In [ ]:
X = train_df.drop("TARGET", axis=1)
y = train_df["TARGET"]

print("Feature Matrix Shape:", X.shape)
print("Target Shape:", y.shape)

## Train Final Deployment Models

Model selection and evaluation have already been completed in the previous notebook.

For deployment, the final selected models are retrained using the complete dataset so they can learn from all available observations.

The following models will be trained:

- XGBoost
- LightGBM

These models will later be combined through probability averaging to form the final ensemble used in production.

In [ ]:
xgb_final = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(y == 0).sum() / (y == 1).sum(),
    random_state=42,
    eval_metric="logloss"
)

xgb_final.fit(X, y)

print("Final XGBoost Training Complete")

In [ ]:
lgbm_final = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    class_weight="balanced",
    random_state=42
)

lgbm_final.fit(X, y)

print("Final LightGBM Training Complete")

## Model Serialization and Persistence

Training machine learning models can be computationally expensive, especially for large datasets and ensemble-based algorithms such as LightGBM and XGBoost.

To avoid retraining the models every time predictions are required, the final trained models are serialized and saved to disk using Joblib.

Benefits of model persistence:

- Eliminates unnecessary retraining
- Reduces application startup time
- Enables deployment in production environments
- Ensures reproducible predictions across sessions
- Allows seamless integration with APIs and web applications

The saved model artifacts will be used in the subsequent inference pipeline and deployment stages of the project.

In [ ]:
joblib.dump(
    lgbm_final,
    "../models/lightgbm_final.pkl"
)

print("LightGBM Saved")

In [ ]:
joblib.dump(
    xgb_final,
    "../models/xgboost_final.pkl"
)

print("XGBoost Saved")

In [ ]:
import os

os.listdir("../models")

## Model Persistence Summary

The final LightGBM and XGBoost models have been successfully trained using the complete processed dataset and saved as serialized artifacts.

Benefits:

- Eliminates the need for retraining during deployment
- Enables rapid model loading in APIs and applications
- Supports scalable production deployment
- Provides reproducible prediction results across environments

These saved models will be used in subsequent sections to build an inference pipeline and risk scoring system.

## Model Loading Verification

Before deployment, it is important to verify that the serialized model artifacts can be successfully loaded back into memory.

This step ensures that the persistence process was completed correctly and that the saved models can be reused for inference without retraining.

In [ ]:
loaded_lgbm = joblib.load("../models/lightgbm_final.pkl")
loaded_xgb = joblib.load("../models/xgboost_final.pkl")

print(type(loaded_lgbm))
print(type(loaded_xgb))

## Preserving the Feature Schema

Machine learning models require the same feature structure during inference that was used during training.

To ensure consistency between training and deployment environments, the complete list of training features is saved separately.

This feature schema will later be used by the API and dashboard layers to:

- Validate incoming customer data
- Preserve feature ordering
- Ensure compatibility between training and inference pipelines
- Prevent prediction failures caused by missing or incorrectly ordered features

Maintaining a dedicated feature schema is a common production machine learning practice and improves the reliability of deployed systems.

In [ ]:
feature_columns = X.columns.tolist()

joblib.dump(
    feature_columns,
    "../models/feature_columns.pkl"
)

print(
    f"{len(feature_columns)} feature names saved successfully"
)

In [ ]:
loaded_features = joblib.load(
    "../models/feature_columns.pkl"
)

print(f"Loaded {len(loaded_features)} features")
print(loaded_features[:10])

## Risk Score Generation

Machine learning models typically produce probabilities between 0 and 1, representing the likelihood of loan default.

While these probabilities are useful for model evaluation, they are not easily interpretable by business stakeholders.

To improve usability, probabilities are converted into a standardized risk score ranging from 0 to 100.

A higher score indicates a greater risk of default.

This risk score serves as a business-friendly representation of model predictions and can be directly displayed in dashboards and decision-support systems.

In [ ]:
def probability_to_risk_score(probability):
    """
    Convert probability (0-1)
    into risk score (0-100)
    """

    return round(probability * 100, 2)

In [ ]:
print(probability_to_risk_score(0.25))
print(probability_to_risk_score(0.50))
print(probability_to_risk_score(0.73))

### Observation

The risk score is a direct transformation of the predicted default probability.

Examples:

| Probability | Risk Score |
|------------|------------|
| 0.25 | 25 |
| 0.50 | 50 |
| 0.73 | 73 |

This representation is significantly easier for business users to interpret than raw probabilities.

## Strategy-Based Risk Categorization

Different financial institutions may prioritize different business objectives when evaluating loan applications.

To simulate real-world decision making, the system supports multiple lending strategies that apply different probability thresholds when categorizing risk.

Available Strategies:

1. Risk-Sensitive Lending Strategy
   - Prioritizes identifying potential defaulters
   - Uses a threshold of 0.50

2. Balanced Operational Strategy
   - Balances risk detection and customer approvals
   - Uses a threshold of 0.55

3. Performance-Optimized Strategy
   - Prioritizes precision and reduction of false alarms
   - Uses a threshold of 0.60

This design separates model predictions from business policy and allows users to explore how different decision strategies affect risk categorization.

In [ ]:
def get_risk_category(probability, strategy="balanced"):

    """
    strategy options:
    - risk_sensitive
    - balanced
    - performance
    """

    thresholds = {
        "risk_sensitive": {
            "medium": 0.25,
            "high": 0.50
        },
        "balanced": {
            "medium": 0.30,
            "high": 0.55
        },
        "performance": {
            "medium": 0.35,
            "high": 0.60
        }
    }

    config = thresholds[strategy]

    if probability < config["medium"]:
        return "Low Risk"

    elif probability < config["high"]:
        return "Medium Risk"

    else:
        return "High Risk"

In [ ]:
#test cell
sample_probability = 0.41

for strategy in [
    "risk_sensitive",
    "balanced",
    "performance"
]:
    
    print(
        f"{strategy}: "
        f"{get_risk_category(sample_probability, strategy)}"
    )

### Observation

A strategy-based risk categorization framework was successfully implemented on top of the model predictions.

Unlike traditional systems that rely on a single fixed threshold, the framework supports multiple business strategies:

| Strategy | High Risk Threshold | Objective |
|-----------|-----------|-----------|
| Risk-Sensitive Lending | 0.50 | Maximize detection of potential defaulters |
| Balanced Operational | 0.55 | Balance risk detection and customer approvals |
| Performance-Optimized | 0.60 | Reduce false alarms and improve precision |

The same applicant may receive different risk classifications under different business strategies while the underlying model prediction remains unchanged.

This design demonstrates the separation of machine learning predictions from business decision policies, a principle commonly used in real-world credit risk systems.

The framework increases flexibility, improves interpretability, and allows stakeholders to evaluate loan applications according to different operational objectives.

## Unified Prediction Engine

To support deployment and application integration, a centralized prediction engine is developed.

The prediction engine combines model inference, risk score generation, and strategy-based risk categorization into a single reusable workflow.

This abstraction layer allows future systems such as REST APIs, web dashboards, and database services to obtain standardized prediction results through a single function call.

The prediction engine serves as the core business logic component of the CreditWise AI platform.

In [ ]:
def predict_customer_risk(
    customer_data,
    strategy="balanced"
):
    
    # LightGBM prediction
    lgbm_prob = loaded_lgbm.predict_proba(
        customer_data
    )[:, 1]
    
    # XGBoost prediction
    xgb_prob = loaded_xgb.predict_proba(
        customer_data
    )[:, 1]
    
    # Ensemble prediction
    final_probability = (
        lgbm_prob + xgb_prob
    ) / 2
    
    probability = float(final_probability[0])
    
    return {
        "default_probability": round(probability, 4),
        "risk_score": probability_to_risk_score(probability),
        "risk_category": get_risk_category(
            probability,
            strategy
        ),
        "strategy": strategy
    }

In [ ]:
#test cell
sample_customer = X.iloc[[2532]]

predict_customer_risk(
    sample_customer,
    strategy="performance"
)

In [ ]:
#strategy comparison test cell
sample_customer = X.iloc[[8556]]

for strategy in [
    "risk_sensitive",
    "balanced",
    "performance"
]:
    
    result = predict_customer_risk(
        sample_customer,
        strategy
    )
    
    print(result)

# Model Explainability using SHAP

Machine learning models such as LightGBM and XGBoost can achieve strong predictive performance but are often considered "black box" models.

To improve transparency and interpretability, SHAP (SHapley Additive exPlanations) is used to explain model predictions.

SHAP values quantify the contribution of each feature to a prediction and help identify the factors driving customer risk.

In financial applications, explainability is particularly important because lending decisions must be transparent, auditable, and understandable by both technical and non-technical stakeholders.

This section explores both global and individual prediction explanations using SHAP.

In [ ]:
import shap
import matplotlib.pyplot as plt

In [ ]:
explainer = shap.TreeExplainer(
    loaded_lgbm
)

print("SHAP Explainer Created")

In [ ]:
sample_data = X.sample(
    n=1000,
    random_state=42
)

shap_values = explainer.shap_values(
    sample_data
)

print("SHAP values generated")

In [ ]:
shap.summary_plot(
    shap_values,
    sample_data,
    plot_type="bar"
)

### Observation

The SHAP global feature importance analysis highlights the factors that contribute most strongly to loan default predictions.

Key findings:

- EXT_SOURCE_MEAN emerged as the most influential feature, confirming the importance of external credit-related information.
- Financial variables such as AMT_CREDIT, AMT_GOODS_PRICE, and AMT_ANNUITY have substantial impact on risk predictions.
- Demographic and employment-related variables contribute additional predictive power.
- Missing value indicators such as EXT_SOURCE_1_MISSING and OWN_CAR_AGE_MISSING appear among the most important features, demonstrating that missingness itself carries useful information.

Overall, the SHAP analysis confirms that the model relies on meaningful financial and behavioral signals rather than arbitrary identifiers or noise variables.

In [ ]:
shap.summary_plot(
    shap_values,
    sample_data
)

### Observation

The SHAP beeswarm plot provides both feature importance and feature effect direction for the model's predictions.

In the visualization:

- Each point represents an individual customer record.
- The horizontal position (SHAP value) indicates how much a feature pushes a prediction toward higher or lower default risk.
- Points on the right side increase predicted default risk.
- Points on the left side decrease predicted default risk.
- Blue points represent lower feature values.
- Red points represent higher feature values.

Key observations include:

- **EXT_SOURCE_MEAN** is the most influential feature in the model. Lower values (blue points) tend to appear on the right side of the plot, increasing default risk, while higher values (red points) appear on the left side, reducing default risk. This aligns with the expected behavior of external creditworthiness indicators.

- **AMT_CREDIT** shows that larger loan amounts generally contribute toward higher default risk, while smaller loan amounts tend to reduce risk.

- **NAME_EDUCATION_TYPE_Higher_education** indicates that applicants with higher education levels are generally associated with lower-risk predictions.

- **DAYS_EMPLOYED** and **EMPLOYMENT_YEARS** suggest that longer and more stable employment histories contribute toward lower default risk, reflecting greater financial stability.

- Missing value indicators such as **EXT_SOURCE_1_MISSING** and **OWN_CAR_AGE_MISSING** continue to influence predictions, demonstrating that the presence or absence of information itself contains useful predictive signals.

Overall, the SHAP analysis confirms that the model relies on meaningful financial, behavioral, and credit-related attributes rather than arbitrary identifiers or noise variables. The observed relationships closely align with real-world lending intuition, increasing confidence in the model's decision-making process.

## Individual Prediction Explanations

In addition to global feature importance, SHAP can explain individual customer predictions.

This allows the system to identify the specific factors contributing to a customer's risk assessment and provides a transparent explanation for each prediction.

Individual explanations are particularly valuable in financial applications where lending decisions must be interpretable and auditable.

In [ ]:
customer_index = 0

customer = X.iloc[[customer_index]]

In [ ]:
customer_shap = explainer.shap_values(customer)

In [ ]:
shap.plots.waterfall(
    shap.Explanation(
        values=customer_shap[0],
        base_values=explainer.expected_value,
        data=customer.iloc[0],
        feature_names=customer.columns
    ),
    max_display=15
)

### Observation

The SHAP waterfall plot explains how individual features contribute to a specific customer's prediction.

The prediction begins at the model's baseline expectation and is progressively adjusted by each feature until the final prediction is reached.

In this example:

- EXT_SOURCE_MEAN is the dominant contributor to risk, adding the largest positive impact to the prediction.
- Additional external credit indicators such as EXT_SOURCE_3 and EXT_SOURCE_1 further increase predicted default risk.
- Financial and behavioral factors including AMT_GOODS_PRICE and social circle default indicators also contribute positively to risk.
- Features such as AGE_YEARS and DAYS_BIRTH partially reduce the predicted risk but are insufficient to offset the influence of the dominant risk factors.

The analysis demonstrates that the model's prediction is not driven by a single variable but rather by the combined influence of multiple financial, demographic, and credit-related attributes.

This level of explainability improves model transparency and provides stakeholders with clear insight into the factors driving an individual lending decision.

## Persisting SHAP Global Feature Importance

To support future product design decisions and recommendation systems, the global SHAP feature importance rankings are persisted for reuse across notebooks and deployment environments.

This avoids recomputing SHAP values repeatedly and establishes a reusable explainability artifact.

In [ ]:
mean_abs_shap = np.abs(
    shap_values
).mean(axis=0)

feature_importance_df = pd.DataFrame({
    "feature": sample_data.columns,
    "importance": mean_abs_shap
})

feature_importance_df = (
    feature_importance_df
    .sort_values(
        by="importance",
        ascending=False
    )
    .reset_index(drop=True)
)

feature_importance_df.head()

In [ ]:
feature_importance_df.to_csv(
    "../data/processed/shap_feature_importance.csv",
    index=False
)

print(
    "SHAP Feature Importance Saved Successfully"
)

## High-Risk vs Low-Risk Applicant Comparison

To further demonstrate model interpretability, SHAP explanations are generated for two extreme cases:

- The applicant with the highest predicted probability of default.
- The applicant with the lowest predicted probability of default.

Comparing these cases provides insight into how the model distinguishes between high-risk and low-risk borrowers and highlights the key factors driving opposite lending decisions.

In [ ]:
# LightGBM probabilities
lgbm_probs = loaded_lgbm.predict_proba(X)[:, 1]

# XGBoost probabilities
xgb_probs = loaded_xgb.predict_proba(X)[:, 1]

# Ensemble probabilities
ensemble_probs = (
    lgbm_probs + xgb_probs
) / 2

In [ ]:
risk_df = pd.DataFrame({
    "default_probability": ensemble_probs
})

risk_df.head()

In [ ]:
high_risk_idx = risk_df["default_probability"].idxmax()

low_risk_idx = risk_df["default_probability"].idxmin()

print(high_risk_idx)
print(low_risk_idx)

high_risk_customer = X.loc[[high_risk_idx]]
low_risk_customer = X.loc[[low_risk_idx]]

In [ ]:
print(
    "High Risk Probability:",
    risk_df.loc[high_risk_idx, "default_probability"]
)

print(
    "Low Risk Probability:",
    risk_df.loc[low_risk_idx, "default_probability"]
)

In [ ]:
high_shap = explainer.shap_values(
    high_risk_customer
)

shap.plots.waterfall(
    shap.Explanation(
        values=high_shap[0],
        base_values=explainer.expected_value,
        data=high_risk_customer.iloc[0],
        feature_names=high_risk_customer.columns
    ),
    max_display=15
)

In [ ]:
low_shap = explainer.shap_values(
    low_risk_customer
)

shap.plots.waterfall(
    shap.Explanation(
        values=low_shap[0],
        base_values=explainer.expected_value,
        data=low_risk_customer.iloc[0],
        feature_names=low_risk_customer.columns
    ),
    max_display=15
)

## High-Risk vs Low-Risk Applicant Analysis

To demonstrate the interpretability of the CreditWise AI model, SHAP explanations were generated for two extreme cases:

- Highest-risk applicant identified by the model
- Lowest-risk applicant identified by the model

### Prediction Summary

| Applicant Type | Customer Index | Default Probability |
|----------------|---------------|---------------------|
| High-Risk Applicant | 57381 | 94.19% |
| Low-Risk Applicant | 104842 | 1.36% |

The substantial difference in predicted risk demonstrates the model's ability to distinguish between highly risky and highly reliable borrowers using financial, demographic, and behavioral information.

### High-Risk Applicant Interpretation

The highest-risk applicant received a predicted default probability of 94.19%.

The dominant risk factors include:

- Extremely low EXT_SOURCE_MEAN, which was the strongest contributor toward default risk.
- Very low external credit scores (EXT_SOURCE_2 and EXT_SOURCE_3).
- High loan-related financial exposure, including AMT_CREDIT, AMT_GOODS_PRICE, and AMT_ANNUITY.
- Short employment history, suggesting limited employment stability.
- Additional behavioral and location-based indicators that further increased risk.

The external credit-related variables alone contributed the majority of the final risk score, indicating that the applicant exhibits characteristics commonly associated with historical loan defaults.

Overall, multiple independent risk signals aligned in the same direction, resulting in a very high predicted probability of default.

### Low-Risk Applicant Interpretation

The lowest-risk applicant received a predicted default probability of only 1.36%.

The strongest factors reducing risk include:

- High EXT_SOURCE_MEAN and strong external credit indicators.
- Higher education status, which contributed toward lower predicted risk.
- Favorable housing and living-condition related attributes.
- Stable organizational and employment-related characteristics.
- Several additional financial and demographic signals that collectively reduced default probability.

Unlike the high-risk applicant, the majority of influential features pushed the prediction toward lower risk, producing a very low probability of default.

The model identified this applicant as financially stable and substantially less likely to default on future loan obligations.

### Comparative Analysis

The SHAP explanations highlight how the model differentiates between high-risk and low-risk borrowers.

Key contrasts observed include:

| Risk Driver | High-Risk Applicant | Low-Risk Applicant |
|-------------|--------------------|--------------------|
| EXT_SOURCE_MEAN | Very Low | High |
| External Credit Indicators | Weak | Strong |
| Financial Exposure | Higher | More Favorable |
| Employment Stability | Limited | Stronger |
| Overall SHAP Direction | Risk Increasing | Risk Decreasing |
| Predicted Default Probability | 94.19% | 1.36% |

The comparison demonstrates that CreditWise AI does not rely on a single feature when generating predictions. Instead, the model evaluates a combination of credit, financial, employment, demographic, and behavioral attributes to arrive at a final risk assessment.

This level of transparency is particularly valuable in financial applications, where stakeholders require both accurate predictions and clear explanations for lending decisions.

# Production Prediction Pipeline

The previous sections developed the individual components required for a production-ready credit risk prediction system.

These components include:

- Ensemble model predictions
- Risk score generation
- Strategy-based threshold selection
- Risk categorization

This section combines all components into a single reusable prediction pipeline.

The resulting function simulates the behavior that will later be exposed through the FastAPI service and consumed by the React dashboard.

## Risk Score Generation

The predicted probability of default is transformed into a risk score ranging from 0 to 100.

A higher score indicates a greater likelihood of loan default and therefore higher lending risk.

In [ ]:
def generate_risk_score(probability):
    """
    Convert probability into
    a 0-100 risk score.
    """
    
    return round(probability * 100, 2)

## Strategy-Based Risk Categorization

Different lending institutions may adopt different risk appetites.

To simulate real-world business scenarios, three lending strategies are supported:

- Risk-Sensitive Lending Strategy
- Balanced Operational Strategy
- Performance-Optimized Strategy

Each strategy uses a different decision threshold while maintaining the same underlying model predictions.

In [ ]:
def get_risk_category(
    probability,
    strategy="balanced"
):
    
    strategy_thresholds = {
        "risk_sensitive": 0.50,
        "balanced": 0.55,
        "performance_optimized": 0.60
    }
    
    threshold = strategy_thresholds[strategy]
    
    if probability >= threshold:
        return "High Risk"
    
    elif probability >= threshold * 0.6:
        return "Medium Risk"
    
    else:
        return "Low Risk"

## Unified Prediction Function

The unified prediction function combines:

- Ensemble prediction
- Risk score calculation
- Strategy-based categorization

into a single reusable workflow.

This function serves as the foundation for future API endpoints and dashboard interactions.

In [ ]:
def predict_customer_risk(
    customer_data,
    strategy="balanced"
):
    
    # LightGBM probability
    lgbm_prob = loaded_lgbm.predict_proba(
        customer_data
    )[:, 1][0]
    
    # XGBoost probability
    xgb_prob = loaded_xgb.predict_proba(
        customer_data
    )[:, 1][0]
    
    # Ensemble probability
    probability = (
        lgbm_prob + xgb_prob
    ) / 2
    
    risk_score = generate_risk_score(
        probability
    )
    
    risk_category = get_risk_category(
        probability,
        strategy
    )
    
    return {
        "probability": round(
            probability,
            4
        ),
        "risk_score": risk_score,
        "risk_category": risk_category,
        "strategy": strategy
    }

## Pipeline Demonstration

The production pipeline is tested using a sample customer record to verify that all components function correctly when integrated together.

In [ ]:
high_risk_result = predict_customer_risk(
    high_risk_customer,
    strategy="balanced"
)

high_risk_result

## Strategy Comparison

The same applicant may be classified differently depending on the organization's risk tolerance.

The following comparison demonstrates how lending strategies affect final risk categorization while keeping the underlying prediction unchanged.

In [ ]:
for strategy in [
    "risk_sensitive",
    "balanced",
    "performance_optimized"
]:
    
    result = predict_customer_risk(
        high_risk_customer,
        strategy=strategy
    )
    
    print("\n", strategy.upper())
    print(result)

In [ ]:
for strategy in [
    "risk_sensitive",
    "balanced",
    "performance_optimized"
]:
    
    result = predict_customer_risk(
        low_risk_customer,
        strategy=strategy
    )
    
    print("\n", strategy.upper())
    print(result)

In [ ]:
borderline_idx = (
    np.abs(ensemble_probs - 0.55)
).argmin()

print(borderline_idx)
print(ensemble_probs[borderline_idx])

borderline_customer = X.iloc[[borderline_idx]]

In [ ]:
for strategy in [
    "risk_sensitive",
    "balanced",
    "performance_optimized"
]:
    
    result = predict_customer_risk(
        borderline_customer,
        strategy=strategy
    )
    
    print("\n", strategy.upper())
    print(result)

### Observation

The production prediction pipeline successfully integrates ensemble inference, risk scoring, and strategy-based risk categorization into a unified workflow.

Testing was performed using three representative applicant profiles:

- Highest-risk applicant identified by the model
- Lowest-risk applicant identified by the model
- Borderline applicant located near the balanced decision threshold

---

### High-Risk Applicant

- Predicted Default Probability: 94.19%
- Risk Score: 94.19/100

The applicant was classified as **High Risk** under all three lending strategies.

This behavior is expected because the predicted probability is significantly higher than every strategy threshold (0.50, 0.55, and 0.60). Changes in business policy therefore do not affect the final lending decision.

---

### Low-Risk Applicant

- Predicted Default Probability: 1.36%
- Risk Score: 1.36/100

The applicant was classified as **Low Risk** under all three lending strategies.

This is also expected because the predicted probability is substantially lower than all decision thresholds.

---

### Borderline Applicant

- Customer Index: 171739
- Predicted Default Probability: 55.00%
- Risk Score: 55.00/100

The applicant was positioned very close to the balanced strategy threshold.

Classification results:

| Strategy | Threshold | Classification |
|-----------|-----------|----------------|
| Risk-Sensitive Lending | 0.50 | High Risk |
| Balanced Operational | 0.55 | High Risk |
| Performance-Optimized | 0.60 | Medium Risk |

This example demonstrates how business strategy influences lending decisions for borderline applicants. While the underlying model prediction remains unchanged, different organizational risk appetites can lead to different final classifications.

---

### Business Interpretation

The results demonstrate that strategy thresholds primarily affect applicants near the decision boundary.

Extremely low-risk and extremely high-risk applicants remain consistently classified regardless of business policy, while borderline applicants may receive different classifications depending on organizational risk tolerance.

This behavior closely reflects real-world credit risk systems where policy settings influence marginal lending decisions without altering the underlying machine learning model.